In [18]:
from scipy.stats import norm
import numpy as np
import math
from scipy.special import gammaln
from scipy.stats import norm
from scipy.integrate import quad

#生成随机旋转矩阵
def generate_random_rotation(d):
    """
    生成一个 d x d 的随机正交矩阵（旋转矩阵）。
    方法：生成高斯随机矩阵，进行 QR 分解，取 Q 矩阵。
    对应论文中的 Π。
    """
    H = np.random.randn(d, d)       # 独立同分布高斯
    Q, R = np.linalg.qr(H)          # QR 分解，Q 是正交矩阵
    # 注意：QR 分解可能产生符号翻转，但不影响旋转性质
    return Q

def optimal_scalar_codebook(b, d):
    """
    生成最优标量量化码本。
    输入：
        b: 比特数 (1,2,3,4,...)
        d: 向量维度 (用于缩放坐标标准差)
    返回：
        码本数组，长度为 2^b，按升序排列。
        [-c, -c/3, c/3, c] 这样的对称码本，具体值根据 b 和 d 计算。
    """
    # 坐标分布的标准差 = 1/√d
    scale = 1.0 / np.sqrt(d)

    # 对于小 b 使用论文精确值
    if b == 1:
        # 最优 1-bit 码本: ± sqrt(2/π)
        c = math.sqrt(2.0 / math.pi)
        return np.array([-c, c]) * scale
    elif b == 2:
        # 论文中给出的近似值: ±0.453, ±1.51
        return np.array([-1.51, -0.453, 0.453, 1.51]) * scale
    else:
        # # 对于 b>=3，使用 Lloyd-Max 算法求解高斯分布的最优量化器
        # # 这里我们采用快速近似：使用等概率分位数初始化，然后迭代优化
        # # 更精确的做法是使用 scipy 的优化或预计算表，为简化，这里用分位数法近似
        # m = 2**b
        # # 标准正态分位数（对称）
        # # 将区间 [0,1] 分成 m 段，取每段的中点的分位数
        # quantiles = np.linspace(0, 1, m + 1)
        # points = np.zeros(m)
        # for i in range(m):
        #     # 区间 [quantiles[i], quantiles[i+1]] 的期望值
        #     # 对于高斯分布，可以使用数值积分，这里用中点近似
        #     # 更准确：使用条件期望公式，但近似已足够
        #     a = quantiles[i]
        #     b_ = quantiles[i+1]
        #     # 用该区间的高斯期望作为码本点
        #     # 对于对称分布，我们可以仅计算正半部分
        #     # 这里简单用分位数的中点，实际效果不差
        #     points[i] = norm.ppf((a + b_) / 2)
        # # 由于分布对称，我们确保码本对称
        # # 调整使均值为0，但分位数法已经对称
        # 标准正态分布下的最优码本，然后缩放

        codebook_std = lloyd_max_gaussian(b)   # 标准正态下的最优码本
        return codebook_std * scale

def lloyd_max_gaussian(b, max_iter=100, tol=1e-6):
    """
    使用 Lloyd-Max 算法求解标准正态分布的最优标量量化码本。
    返回码本数组，长度为 2^b。
    """
    m = 2**b
    # 正确初始化：将 [0,1] 分成 m 个等概率区间，每个区间的代表点为该区间的中点分位数
    quantiles = np.linspace(0, 1, m + 1)                # m+1 个分位点
    codebook = norm.ppf((quantiles[1:] + quantiles[:-1]) / 2)   # 取每个区间的中点分位数，得到 m 个初始码本
    for _ in range(max_iter):
        # 计算分区边界（相邻码本的中点）
        boundaries = (codebook[:-1] + codebook[1:]) / 2
        boundaries = np.concatenate(([-np.inf], boundaries, [np.inf]))
        new_codebook = []
        for i in range(m):
            a, b_ = boundaries[i], boundaries[i + 1]
            # 计算区间 [a, b_] 内的条件期望 ∫ x f(x) dx / ∫ f(x) dx
            num, _ = quad(lambda x: x * norm.pdf(x), a, b_)
            den, _ = quad(norm.pdf, a, b_)
            if den > 0:
                new_codebook.append(num / den)
            else:
                new_codebook.append(codebook[i])  # 避免除以零
        new_codebook = np.array(new_codebook)
        # 检查收敛
        if np.max(np.abs(new_codebook - codebook)) < tol:
            break
        codebook = new_codebook
    return codebook


class TurboQuantMSE:
    """MSE 优化的 TurboQuant 量化器"""
    def __init__(self, d, b):
        self.d = d
        self.b = b
        self.Pi = generate_random_rotation(d)
        self.codebook = optimal_scalar_codebook(b, d)
        # 为了快速查找，可以预先计算码本边界（相邻质心的中点），但我们使用最近邻搜索
        # 由于码本规模小 (最多 2^4=16)，直接计算距离即可。

    def quantize(self, x):
        """
        对应论文 Algorithm 1 中的 QUANTmse。
        输入: x (d维向量)
        返回: idx (长度为 d 的整数数组)
        """
        # 旋转
        y = self.Pi @ x
        # 对每个坐标量化
        idx = np.zeros(self.d, dtype=int)
        for j in range(self.d):
            # 找到最近码本
            diff = np.abs(y[j] - self.codebook) # 计算与码本每个点的距离
            idx[j] = np.argmin(diff) # 选择最近的码本索引
        return idx

    def dequantize(self, idx):
        """
        对应论文 Algorithm 1 中的 DEQUANTmse。
        输入: idx (长度为 d 的整数数组)
        返回: 重建向量 x_hat
        """
        y_hat = self.codebook[idx] # 将索引映射回码本值
        x_hat = self.Pi.T @ y_hat # 逆旋转
        return x_hat # 返回重建向量

    def compress(self, vectors):
        """压缩一批向量"""
        return [self.quantize(v) for v in vectors]

    def decompress(self, compressed):
        """解压一批向量"""
        return np.array([self.dequantize(idx) for idx in compressed])

class QJL:
    """1-bit 内积无偏量化器"""
    def __init__(self, d):
        self.d = d
        # 随机矩阵 S，元素独立标准正态
        self.S = np.random.randn(d, d)

    def quantize(self, x):
        """返回符号向量 (±1)"""
        # 注意：论文中 sign(0) 定义为 1 或 -1,通常定义为 1
        # 这里直接使用 np.sign，0 返回 0，但为了二进制，处理为 1
        # 但实际正态分布下，概率 0 可忽略
        return np.sign(self.S @ x)

    def dequantize(self, z, gamma=1.0):
        """
        反量化，可传入缩放因子 gamma（用于残差长度）
        论文公式：x_hat = (sqrt(pi/2)/d) * gamma * S^T * z
        """
        coef = math.sqrt(math.pi / 2) / self.d * gamma
        return coef * (self.S.T @ z)

class TurboQuantProd:
    """内积优化的 TurboQuant 量化器 (两阶段)"""
    def __init__(self, d, b):
        """
        d: 维度
        b: 总比特数 per coordinate (实际使用 b-1 比特 MSE + 1 比特 QJL)
        """
        self.d = d
        self.b = b
        # 第一阶段：b-1 比特 MSE 量化器
        self.mse_quant = TurboQuantMSE(d, b-1) if b > 1 else None
        # 第二阶段：QJL 1-bit 量化器
        self.qjl = QJL(d)

    def quantize(self, x):
        """
        对应论文 Algorithm 2 中的 QUANTprod。
        输出： (idx, qjl, gamma)
        """
        if self.b == 1:
            # 特例：只使用 QJL 量化整个向量
            gamma = np.linalg.norm(x)
            if gamma > 0:
                x_unit = x / gamma
            else:
                x_unit = np.zeros(self.d)
            qjl = self.qjl.quantize(x_unit)
            return (None, qjl, gamma)
        else:
            # 第一步：b-1 比特 MSE 量化
            idx = self.mse_quant.quantize(x)
            x_mse = self.mse_quant.dequantize(idx)
            # 计算残差
            r = x - x_mse
            gamma = np.linalg.norm(r)  #将残差保存，用于后续缩放,这里需要注意的是QJL 的理论保证（无偏性、方差界）是基于单位向量 ，所以需要对残差分解为 长度 和 方向，并将长度作为缩放因子传递给 QJL 的反量化函数，以确保最终的重建向量在内积意义上接近原始向量。
            # 第二步：对残差单位向量应用 QJL
            if gamma > 0:
                r_unit = r / gamma #如果残差>0，则归一化，否则保持为零向量（避免除以零）
            else:
                r_unit = np.zeros(self.d)
            qjl = self.qjl.quantize(r_unit) # 对残差的单位方向向量进行 1-bit 量化，用sign函数得到 ±1 的符号向量
            return (idx, qjl, gamma)

    def dequantize(self, idx, qjl, gamma):
        """
        对应论文 Algorithm 2 中的 DEQUANTprod。
        """
        if self.b == 1:
            # 只有 QJL 部分
            return self.qjl.dequantize(qjl, gamma)
        else:
            x_mse = self.mse_quant.dequantize(idx)
            x_qjl = self.qjl.dequantize(qjl, gamma)
            return x_mse + x_qjl

    def compress(self, vectors):
        """压缩一批向量"""
        return [self.quantize(v) for v in vectors]

    def decompress(self, compressed):
        """解压一批向量"""
        return np.array([self.dequantize(idx, qjl, gamma) for (idx, qjl, gamma) in compressed])

def inner_product_estimation(tq_prod, query, compressed):
    """
    使用 TurboQuantProd 计算查询向量与压缩数据库向量的内积估计。
    参数：
        tq_prod: TurboQuantProd 实例
        query: 查询向量 (d 维)
        compressed: 压缩后的数据库列表，每个元素为 (idx, qjl, gamma)
    返回：
        内积估计数组
    """
    # 对于 MSE 部分，我们旋转查询向量以与旋转后的坐标对齐
    if tq_prod.b > 1:
        # 旋转查询向量
        query_rot = tq_prod.mse_quant.Pi @ query
    else:
        query_rot = None

    estimates = []
    for (idx, qjl, gamma) in compressed:
        if tq_prod.b == 1:
            # 只使用 QJL 部分
            # 内积 = (sqrt(pi/2)/d) * gamma * (qjl^T * (S @ query))
            Sz = tq_prod.qjl.S @ query
            inner = (math.sqrt(math.pi / 2) / tq_prod.d) * gamma * np.dot(qjl, Sz)
            estimates.append(inner)
        else:
            # 第一部分：MSE 重建向量与查询的内积 = 旋转后坐标的点积
            y_hat = tq_prod.mse_quant.codebook[idx]  # 旋转后的重建向量
            inner1 = np.dot(query_rot, y_hat)
            # 第二部分：QJL 部分
            Sz = tq_prod.qjl.S @ query
            inner2 = (math.sqrt(math.pi / 2) / tq_prod.d) * gamma * np.dot(qjl, Sz)
            estimates.append(inner1 + inner2)
    return np.array(estimates)




In [19]:
# 参数设置
d = 128          # 向量维度
b = 6            # 每坐标比特数（对于 prod 模式，实际 b=3 表示 2-bit MSE + 1-bit QJL）
n = 10000        # 数据库大小

# 生成随机数据库向量（假设已归一化，实际可存储范数）
np.random.seed(42)
db_vectors = np.random.randn(n, d)
db_vectors = db_vectors / np.linalg.norm(db_vectors, axis=1, keepdims=True)

for b in range(2,10):
    # 初始化 TurboQuantProd
    tq = TurboQuantProd(d, b)

    # 压缩数据库
    compressed_db = tq.compress(db_vectors)

    # 存储大小评估
    if b == 1:
        bits_per_vector = d + 32   # qjl 的 d 比特 + gamma 浮点数
    else:
        bits_per_vector = d * (b-1) + d + 32  # idx (d*(b-1) bits) + qjl (d bits) + gamma
    original_bits = d * 32
    print(f"原始每向量: {original_bits} 比特")
    print(f"压缩后每向量: {bits_per_vector} 比特")
    print(f"压缩比: {original_bits / bits_per_vector:.2f}x")

    # 查询向量
    query = np.random.randn(d)
    query = query / np.linalg.norm(query)

    # 计算内积估计
    inner_est = inner_product_estimation(tq, query, compressed_db)

    # 真实内积
    true_inner = db_vectors @ query

    # 误差分析
    mse = np.mean((true_inner - inner_est)**2)
    print(f"内积估计 MSE: {mse:.6f}")

    # 召回率评估 (Top-10)
    topk = 10
    true_topk = np.argsort(-true_inner)[:topk]
    est_topk = np.argsort(-inner_est)[:topk]
    recall = len(set(true_topk) & set(est_topk)) / topk
    print(f"Recall@{topk}: {recall:.4f}")
    print("------------------------------")

原始每向量: 4096 比特
压缩后每向量: 288 比特
压缩比: 14.22x
内积估计 MSE: 0.004391
Recall@10: 0.3000
------------------------------
原始每向量: 4096 比特
压缩后每向量: 416 比特
压缩比: 9.85x
内积估计 MSE: 0.001371
Recall@10: 0.5000
------------------------------
原始每向量: 4096 比特
压缩后每向量: 544 比特
压缩比: 7.53x
内积估计 MSE: 0.000407
Recall@10: 0.7000
------------------------------
原始每向量: 4096 比特
压缩后每向量: 672 比特
压缩比: 6.10x
内积估计 MSE: 0.000150
Recall@10: 0.9000
------------------------------
原始每向量: 4096 比特
压缩后每向量: 800 比特
压缩比: 5.12x
内积估计 MSE: 0.000031
Recall@10: 0.9000
------------------------------
原始每向量: 4096 比特
压缩后每向量: 928 比特
压缩比: 4.41x
内积估计 MSE: 0.000011
Recall@10: 0.8000
------------------------------
原始每向量: 4096 比特
压缩后每向量: 1056 比特
压缩比: 3.88x
内积估计 MSE: 0.000004
Recall@10: 1.0000
------------------------------
原始每向量: 4096 比特
压缩后每向量: 1184 比特
压缩比: 3.46x
内积估计 MSE: 0.000002
Recall@10: 1.0000
------------------------------


- 发现召回率并不是随着 b 单调递增，而是在某些 b 出现波动。
  1. 码本近似误差：对于 b≥3，使用了分位数近似码本，而不是真正的 Lloyd-Max 最优码本。这可能导致某些 b 的量化误差并非最优，影响内积估计精度。
  2. 样本数：10000 个数据库向量可能不足以平滑掉随机波动。

In [20]:
def t2():
    # 设置参数
    d = 128
    n = 10000
    n_queries = 100   # 使用多个查询取平均
    n_runs = 5        # 重复实验次数

    # 固定随机种子，生成固定的数据库和查询集
    np.random.seed(42)
    db_vectors = np.random.randn(n, d)
    db_vectors = db_vectors / np.linalg.norm(db_vectors, axis=1, keepdims=True)

    queries = np.random.randn(n_queries, d)
    queries = queries / np.linalg.norm(queries, axis=1, keepdims=True)

    # 存储结果
    results = {b: {'mse': [], 'recall': []} for b in range(2, 10)}

    for run in range(n_runs):
        for b in range(2, 10):
            tq = TurboQuantProd(d, b)  # 注意：每次运行重新生成随机旋转和 QJL 矩阵
            compressed_db = tq.compress(db_vectors)

            # 对每个查询计算
            recalls = []
            mses = []
            for q in queries:
                inner_est = inner_product_estimation(tq, q, compressed_db)
                true_inner = db_vectors @ q
                mse = np.mean((true_inner - inner_est)**2)
                mses.append(mse)
                # 计算召回率
                topk = 10
                true_topk = np.argsort(-true_inner)[:topk]
                est_topk = np.argsort(-inner_est)[:topk]
                recall = len(set(true_topk) & set(est_topk)) / topk
                recalls.append(recall)

            results[b]['mse'].append(np.mean(mses))
            results[b]['recall'].append(np.mean(recalls))
            print(f"第{b}的结果是 mse 是{np.mean(mses)} .recall 是{np.mean(recalls)}")


    # 汇总平均
    for b in range(2, 10):
        mean_mse = np.mean(results[b]['mse'])
        std_mse = np.std(results[b]['mse'])
        mean_recall = np.mean(results[b]['recall'])
        std_recall = np.std(results[b]['recall'])
        print(f"b={b}: MSE={mean_mse:.6f}±{std_mse:.6f}, Recall={mean_recall:.4f}±{std_recall:.4f}")

t2()

第2的结果是 mse 是0.004399606555520542 .recall 是0.259
第3的结果是 mse 是0.0013852227367905305 .recall 是0.47400000000000014
第4的结果是 mse 是0.0004216766612072311 .recall 是0.6579999999999998
第5的结果是 mse 是0.00011539908214567765 .recall 是0.8079999999999999
第6的结果是 mse 是3.2861934771619176e-05 .recall 是0.8980000000000002
第7的结果是 mse 是1.1990802734975299e-05 .recall 是0.9390000000000004
第8的结果是 mse 是5.008588181840344e-06 .recall 是0.9670000000000002
第9的结果是 mse 是1.9478789310271384e-06 .recall 是0.9700000000000002
第2的结果是 mse 是0.0044474206069080235 .recall 是0.266
第3的结果是 mse 是0.0014286866619310193 .recall 是0.473
第4的结果是 mse 是0.00041779846836032355 .recall 是0.66
第5的结果是 mse 是0.0001161096725583487 .recall 是0.8150000000000003
第6的结果是 mse 是3.287872172177668e-05 .recall 是0.9150000000000001
第7的结果是 mse 是1.235679979361275e-05 .recall 是0.9510000000000002
第8的结果是 mse 是4.718239629395167e-06 .recall 是0.9590000000000002
第9的结果是 mse 是1.919986147272908e-06 .recall 是0.9810000000000002
第2的结果是 mse 是0.004237663970528457 .recall 是0.26
第3的结果是 ms